# Frankie NSFW Generator — Pony XL + IPAdapter FaceID

**One task**: generate 8 uncensored Frankie selfies for internal MVP review.

Runtime → **Run all**. Walk away ~10 min. Images appear at the bottom.

Requires: Free Colab T4 (Runtime → Change runtime type → T4 GPU).

In [ ]:
# ── 1. GPU check ──
!nvidia-smi | head -15

In [ ]:
# ── 2. Install (pinned versions — avoids bleeding-edge breakage) ──
!pip install -q diffusers==0.32.2 transformers==4.46.3 accelerate==1.1.1 peft==0.13.2 safetensors
!pip install -q pillow insightface onnxruntime-gpu einops huggingface_hub
import diffusers, transformers, peft
print(f'diffusers {diffusers.__version__} | transformers {transformers.__version__} | peft {peft.__version__}')

In [ ]:
# ── 3. Download Pony Diffusion v6 XL ──
import os
os.makedirs('/content/models', exist_ok=True)
if not os.path.exists('/content/models/pony_v6.safetensors'):
    !wget -q --show-progress -O /content/models/pony_v6.safetensors 'https://huggingface.co/AstraliteHeart/pony-diffusion-v6/resolve/main/v6.safetensors'
!ls -lh /content/models/

In [ ]:
# ── 4. Download IPAdapter FaceID + CLIP-ViT-H encoder ──
!mkdir -p /content/models/ipadapter /content/models/image_encoder
if not os.path.exists('/content/models/ipadapter/ip-adapter-faceid-plusv2_sdxl.bin'):
    !wget -q --show-progress -O /content/models/ipadapter/ip-adapter-faceid-plusv2_sdxl.bin 'https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl.bin'
    !wget -q --show-progress -O /content/models/ipadapter/ip-adapter-faceid-plusv2_sdxl_lora.safetensors 'https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl_lora.safetensors'
from huggingface_hub import snapshot_download
snapshot_download(repo_id='laion/CLIP-ViT-H-14-laion2B-s32B-b79K', local_dir='/content/models/image_encoder', allow_patterns=['*.json','*.bin','*.safetensors'])

In [ ]:
# ── 5. Load Frankie reference ──
import requests, io
from PIL import Image
FRANKIE_REF_URL = 'https://raw.githubusercontent.com/veyrapay/frankie-assets/main/frankie-ref.png'
frankie_img = Image.open(io.BytesIO(requests.get(FRANKIE_REF_URL).content)).convert('RGB')
display(frankie_img.resize((400, 327)))
print(f'Frankie reference: {frankie_img.size}')

In [ ]:
# ── 6. Extract face embedding via InsightFace ──
import cv2, numpy as np, torch
from insightface.app import FaceAnalysis
face_app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider','CPUExecutionProvider'])
face_app.prepare(ctx_id=0, det_size=(640, 640))
frankie_cv = cv2.cvtColor(np.array(frankie_img), cv2.COLOR_RGB2BGR)
faces = face_app.get(frankie_cv)
assert len(faces) > 0, 'no face detected in reference'
face_embed = torch.from_numpy(faces[0].normed_embedding).unsqueeze(0)
print(f'Face embed: {face_embed.shape}')

In [ ]:
# ── 7. Load Pony XL pipeline with IPAdapter FaceID (diffusers native) ──
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
from transformers import CLIPVisionModelWithProjection, CLIPImageProcessor
image_encoder = CLIPVisionModelWithProjection.from_pretrained('/content/models/image_encoder', torch_dtype=torch.float16).to('cuda')
feature_extractor = CLIPImageProcessor()
pipe = StableDiffusionXLPipeline.from_single_file(
    '/content/models/pony_v6.safetensors',
    torch_dtype=torch.float16,
    image_encoder=image_encoder,
    feature_extractor=feature_extractor,
).to('cuda')
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
pipe.load_ip_adapter('/content/models/ipadapter', subfolder='', weight_name='ip-adapter-faceid-plusv2_sdxl.bin', image_encoder_folder=None)
pipe.set_ip_adapter_scale(0.8)
pipe.load_lora_weights('/content/models/ipadapter/ip-adapter-faceid-plusv2_sdxl_lora.safetensors')
pipe.fuse_lora(lora_scale=0.7)
print('Pipeline ready')

In [ ]:
# ── 8. Generate Frankie scenes ──
POSITIVE_PREFIX = 'score_9, score_8_up, score_7_up, source_photo, photorealistic, sharp focus, natural skin texture, freckles, dark wavy hair, gold hoop earrings, \"Frankie\" name necklace, sleeve tattoos, '
NEGATIVE = 'score_4, score_5, score_6, source_furry, source_anime, source_cartoon, source_pony, deformed, bad anatomy, extra fingers, watermark, text, signature, blurry, low quality, ugly'
SCENES = [
    'topless mirror selfie in bathroom, soft warm lighting, intimate',
    'topless in bed in morning light, smiling, candid',
    'wearing black lace lingerie, sitting on bed, looking at camera, sultry',
    'in a white t-shirt and panties, kitchen morning, holding coffee',
    'topless on a beach at sunset, ocean behind, hair wet',
    'in a sheer black robe, balcony at night, city lights behind',
    'taking a selfie in a black string bikini, pool behind, sun-kissed',
    'in a tight white tank top with no bra, gym mirror selfie, sweaty',
]
os.makedirs('/content/out', exist_ok=True)
results = []
face_embed_cuda = face_embed.to('cuda', torch.float16).unsqueeze(0)
for i, scene in enumerate(SCENES, 1):
    prompt = POSITIVE_PREFIX + scene
    print(f'\n[{i}/{len(SCENES)}] {scene}')
    gen = torch.Generator('cuda').manual_seed(42 + i)
    img = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE,
        ip_adapter_image_embeds=[face_embed_cuda],
        width=1024, height=1024,
        num_inference_steps=30,
        guidance_scale=6.0,
        generator=gen,
    ).images[0]
    out_path = f'/content/out/frankie_{i:02d}.png'
    img.save(out_path)
    results.append((scene, img, out_path))

In [ ]:
# ── 9. Display all results ──
from IPython.display import display, Markdown
for scene, img, path in results:
    display(Markdown(f'**{scene}**'))
    display(img.resize((512, 512)))

In [ ]:
# ── 10. Zip & download ──
!cd /content/out && zip -q /content/frankie_nsfw_batch.zip *.png
from google.colab import files
files.download('/content/frankie_nsfw_batch.zip')